In [ ]:
import numpy as np
import pandas as pd
import json
import yaml
from tqdm import tqdm

In [ ]:
import os
os.chdir('../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

### 1. CSMAR symbol with subs name and the WOS affiliation

In [ ]:
Qichacha_sbus_eng_high = pd.read_csv(dataset_config['path_processed'] + 'WOS_listed/CN_subs_WOS_highmatch.csv')
Qichacha_sbus_eng_high

In [ ]:
symbol_subs = pd.read_csv(dataset_config['path_processed'] + "WOS_listed/CSMAR_subs_name.csv")
symbol_subs.rename(columns={'RalatedParty': 'CSMAR_subs_name'}, inplace=True)
symbol_subs

In [ ]:
symbol_WOSaff = pd.merge(symbol_subs, Qichacha_sbus_eng_high, on='CSMAR_subs_name')
symbol_WOSaff

### 2. matched WOS id

In [ ]:
aff_wosid = pd.read_parquet(dataset_config['path_processed'] + 'WOS/WOS_paperid_CNfirm.parquet')
aff_wosid

In [ ]:
aff_wos_in_subs = aff_wosid.merge(
    Qichacha_sbus_eng_high[['affiliationame_match']],   # keep only the matching column
    left_on='affiliationame',
    right_on='affiliationame_match',
    how='inner'
)
aff_wos_in_subs

#### Merge with year

In [ ]:
wos_paper_info = pd.read_parquet(dataset_config['path_processed'] + 'WOS/WOS_paper_level.parquet').drop_duplicates()
wos_paper_info.rename(columns={'pub year': 'year'}, inplace=True)
wos_paper_info

In [ ]:
aff_wos_in_subs_withyear = pd.merge(aff_wos_in_subs, wos_paper_info, on='wosid')
aff_wos_in_subs_withyear

### 3. merge 1 and 2

In [ ]:
symbol_subs_wosid = pd.merge(symbol_WOSaff, aff_wos_in_subs_withyear, on='affiliationame_match')
symbol_subs_wosid

In [ ]:
df_sub = symbol_subs_wosid[['Symbol', 'wosid', 'year', 'fractional_num']].drop_duplicates()
df_count = (
    df_sub
    .groupby(['Symbol', 'year'])
    .agg(
        subs_add=('wosid', 'nunique'),  # count unique wosid
        subs_frac_add=('fractional_num', 'sum')  # sum of fractional_num
    )
    .reset_index()
)
df_count

In [ ]:
df_count.to_csv(dataset_config['path_processed'] + 'WOS_listed/CN_listed_subs_add.csv', index=False)